In [1]:
import matplotlib.pyplot as plt
%matplotlib widget
import json
from pprint import pprint
import scipy.io
import numpy as np
from pathlib import Path
import spikeinterface as si  # import core only
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.postprocessing as spost
import spikeinterface.qualitymetrics as sqm
import spikeinterface.comparison as sc
import spikeinterface.exporters as sexp
import spikeinterface.curation as scur
import spikeinterface.widgets as sw
from pathlib import Path
import re
import os
import probeinterface as pi
import probeinterface.plotting as pi_plot
import cfad
from cfad import load_mice 
import pandas as pd

global_job_kwargs = dict(n_jobs=4, chunk_duration="1s")
si.set_global_job_kwargs(**global_job_kwargs)


In [ ]:
paths = cfad.data

renewal_neural = paths["Neural"]["Renewal"]
checkerboard_rec = renewal_neural["Raw Data"]["Checkerboard"]
concat_path = renewal_neural["Concatenated Data"]
concat_kilosort = renewal_neural["Concatenated Data"]["kilosort4"]
checkerboard_triggers = renewal_neural["Triggers"]["Checkerboard"]
dtype = "int16"
fs = 30000
num_channels = 385






In [2]:
recordings_checkerboard, sortings_checkerboard, triggers_checkerboard = load_mice(
    start = 2,
    end = 9,
    rec = "Checkerboard",
    phase = "Renewal",
    slice = True
)

recording_habituation, sorting_habituation, triggers_habituation = cfad.load_mice(
    2,
    9,
    "Habituation",
    "Renewal"
)


Mouse 2
0.1949999928474426
Loaded
Triggers from 88362167 to 97430254
FrameSliceRecording: 384 channels - 30.0kHz - 1 segments - 9,188,087 samples 
                     306.27s (5.10 minutes) - int16 dtype - 6.57 GiB
recording start at 2943.405566666667 seconds
recording end at 3249.6751000000004 seconds
Trigger start at 2945.405566666667 seconds
Trigger final at 3247.675133333333 seconds
FrameSliceSorting: 273 units - 1 segments - 30.0kHz
Probe - 384ch - 1shanks


Mouse 3
0.1949999928474426
Loaded
Triggers from 90415683 to 99478843
FrameSliceRecording: 384 channels - 30.0kHz - 1 segments - 9,183,160 samples 
                     306.11s (5.10 minutes) - int16 dtype - 6.57 GiB
recording start at 3011.8561 seconds
recording end at 3317.9614 seconds
Trigger start at 3013.8561 seconds
Trigger final at 3315.9614333333334 seconds
FrameSliceSorting: 235 units - 1 segments - 30.0kHz
Probe - 384ch - 1shanks


Mouse 4
0.1949999928474426


KeyboardInterrupt: 

In [ ]:
import spikeinterface.extractors as se
import numpy as np
import matplotlib.pyplot as plt
import os
from ipywidgets import interact, IntSlider
import ipywidgets as widgets

def lfp_c_p(raw, ks, triggers, concat, ds, lfpr):
    # --- 1. SETUP ---
    raw_path = raw
    ks_path = ks
    
    fs = 30000 / lfpr
    dtype = "int16"
    num_channels = 384
    
    # Parameters
    pre_time = 0.1
    post_time = 0.5
    downsample_factor = ds
    
    # --- 2. LOAD DATA ---
    if concat:
        rec_raw = raw
        triggers_post = triggers
    else:
        rec_raw = se.read_openephys(raw)
        offset = 0
        for i in Path(raw).parents[0].iterdir():
            if i != raw:
                rec_temp = se.read_openephys(raw)
                offset += rec_temp.get_num_frames()
        triggers_post = triggers - offset
        
    rec_raw = rec_raw.rename_channels(np.arange(rec_raw.get_num_channels()))
    channel_map = np.load(os.path.join(ks_path, 'channel_map.npy')).flatten()
    rec = rec_raw.select_channels(channel_map)
    fs = rec.get_sampling_frequency()
    
    # --- 3. SORT CHANNELS BY DEPTH ---
    chan_pos = np.load(os.path.join(ks_path, 'channel_positions.npy'))
    depths = chan_pos[:, 1]
    original_ids = rec.channel_ids
    
    sorted_indices = np.argsort(depths)
    sorted_depths = depths[sorted_indices]
    sorted_ids = original_ids[sorted_indices]
    
    print(f"Data loaded: {len(depths)} channels sorted by depth.")
    
    # --- 4. PROCESS TRIGGERS (1-150) ---
    subset_triggers = np.array(triggers_post[1:151], dtype=int)
    
    if subset_triggers[0] > rec.get_num_frames():
        offset = subset_triggers[0] - (1.0 * fs) 
        local_triggers = subset_triggers - offset
    else:
        local_triggers = subset_triggers
    
    valid_mask = (local_triggers > (pre_time * fs)) & (local_triggers < (rec.get_num_frames() - post_time * fs))
    final_triggers = local_triggers[valid_mask]
    
    # --- 5. EXTRACT ALL TRIALS ---
    print(f"Extracting {len(final_triggers)} trials for SEM calculation...")
    
    num_samples = int((pre_time + post_time) * fs)
    num_ds = num_samples // downsample_factor
    
    # List to store every single trial: [ (N_ch, N_time), (N_ch, N_time), ... ]
    all_trials_list = []
    
    for trig in final_triggers:
        start = int(trig - pre_time * fs)
        end = int(trig + post_time * fs)
        trace = rec.get_traces(start_frame=start, end_frame=end).T
        
        # Downsample
        trace_ds = trace[:, ::downsample_factor]
        
        # Ensure correct shape before appending
        if trace_ds.shape[1] == num_ds:
            all_trials_list.append(trace_ds)
    
    # Convert to 3D Array: (N_trials, N_channels, N_timepoints)
    all_trials = np.array(all_trials_list)
    
   
    # --- 6. BASELINE CORRECTION & STATISTICS ---
    time_axis = np.linspace(-pre_time, post_time, num_ds)
    zero_idx = np.abs(time_axis - 0).argmin()
    
    # Subtract the value at t=0 for EACH trial individually
    # Value at zero: (N_trials, N_channels, 1)
    baseline_vals = all_trials[:, :, zero_idx][:, :, np.newaxis]
    all_trials_norm = all_trials - baseline_vals
    
    # Calculate Mean and SEM across the "Trials" axis (axis 0)
    mean_trace = np.mean(all_trials_norm, axis=0) # (N_channels, N_time)
    std_trace = np.std(all_trials_norm, axis=0)
    sem_trace = std_trace / np.sqrt(len(all_trials_list))
    
    # Sort by Depth
    mean_sorted = mean_trace[sorted_indices, :]
    sem_sorted = sem_trace[sorted_indices, :]
    
    print("Statistics complete. Generating Widget...")
    
    # --- 7. INTERACTIVE PLOTTER ---
    
    def plot_channel(index):
        """
        Plots Mean +/- SEM for a single channel.
        """
        # Get data
        mean = mean_sorted[index]
        sem = sem_sorted[index]
        depth = sorted_depths[index]
        orig_id = sorted_ids[index]
        
        plt.figure(figsize=(10, 5))
        
        # Plot Mean Line
        plt.plot(time_axis, mean, color='black', linewidth=1.5, label='Mean Response')
        
        # Plot SEM Shading
        plt.fill_between(time_axis, mean - sem, mean + sem, 
                         color='black', alpha=0.2, label='SEM')
        
        # Add Markers
        plt.axvline(0, color='red', linestyle='--', alpha=0.5, label='Trigger Onset')
        plt.axhline(0, color='blue', linewidth=1, linestyle='-', alpha=0.2)
        
        # Styling
        plt.title(f"Channel Index: {index} (Orig ID: {orig_id})\nDepth: {depth} $\mu m$ (Normalized)", fontsize=14)
        plt.xlabel("Time from Trigger (s)")
        plt.ylabel("Voltage ($\mu V$)")
        plt.xlim(-pre_time, post_time)
        
        # Dynamic Y-Limits (include the SEM in the bounds calculation)
        upper_bound = np.max(mean + sem)
        lower_bound = np.min(mean - sem)
        buffer = (upper_bound - lower_bound) * 0.1
        if buffer == 0: buffer = 10
        
        plt.ylim(lower_bound - buffer, upper_bound + buffer)
        
        plt.legend(loc='upper right')
        plt.grid(True, alpha=0.3)
        plt.show()
    
    interact(
        plot_channel, 
        index=IntSlider(
            min=0, 
            max=len(sorted_indices)-1, 
            step=1, 
            value=0, 
            description='Sort Index:',
            continuous_update=False 
        )
    );
    
        


In [ ]:
"""
for i in range(2, 10):
    print("Mouse " + str(i))
    lfp_c_p(
        recordings_checkerboard[i],
        concat_kilosort["Mouse " + str(i)],
        triggers_checkerboard[i],
        True
    )
       
    lfp_c_p(
        checkerboard_rec["Mouse " + str(i)],
        concat_kilosort["Mouse " + str(i)],
        triggers_checkerboard[i],
        False
    )
    
"""

In [ ]:
recording = recordings_checkerboard[2]
gain_path = checkerboard_rec["Mouse 2"] + r"\Record Node 101\experiment1\recording1"
def load_gain_from_oebin(recording_folder):
    """
    Parses structure.oebin to find the bit_volts (gain).
    Accepts the folder path and automatically appends 'structure.oebin'.
    """
    # 1. Construct the full file path
    oebin_path = os.path.join(recording_folder, "structure.oebin")
    
    # 2. Check if the FILE exists (not just the folder)
    if not os.path.isfile(oebin_path):
        # Fallback: check one level up (sometimes it sits in 'experiment1')
        parent_dir = os.path.dirname(recording_folder)
        parent_oebin = os.path.join(parent_dir, "structure.oebin")
        if os.path.isfile(parent_oebin):
            oebin_path = parent_oebin
        else:
            raise FileNotFoundError(f"structure.oebin not found at: {oebin_path}")

    # 3. Load and Parse
    with open(oebin_path, 'r') as f:
        meta = json.load(f)
    
    # Look for the Neural stream (usually has the most channels)
    # We loop through to find the one matching your channel count best, or just the largest
    neural_gain = None
    max_channels = 0
    
    for stream in meta.get('continuous', []):
        n_chans = stream.get('num_channels', 0)
        if n_chans > max_channels:
            max_channels = n_chans
            # Grab bit_volts from the first channel in the list
            if len(stream['channels']) > 0:
                neural_gain = stream['channels'][0]['bit_volts']
    
    if neural_gain is None:
        raise ValueError("Could not find a valid neural stream in structure.oebin")
    print(neural_gain)
    return neural_gain
try:
    current_gain = load_gain_from_oebin(gain_path)
    print(f"Loaded gain from metadata: {current_gain} uV/bit")
except Exception as e:
    print(f"Metadata load failed: {e}. Falling back to default NP2.0 gain.")
    current_gain = 2.34375


n_channels = recording.get_num_channels()
recording.set_channel_gains(current_gain)
#recording.set_property("offset_to_uV", np.ones(n_channels) * 0.0)

# 2. Now run the pipeline
# Note: Since you likely have wideband data (containing both spikes and LFP), 
# downsampling is definitely required to isolate the LFP.

#bad_channel_ids, _ = spre.detect_bad_channels(
    #recording, 
    #method="std", 
    #std_mad_threshold=5
#)
#recording_clean = #spre.interpolate_bad_channels(recording, bad_channel_ids)
#print(bad_channel_ids)
# 3. Process to LFP
#recording_lfp = #spre.resample(recording_clean, resample_rate=1000)
recording_lfp_filtered = spre.bandpass_filter(recording, freq_min=0.5, freq_max=300, dtype='int16')
recording_final = spre.common_reference(recording_lfp_filtered, reference='global', operator='median')

# 5. Save/Export
# Now you can save the processed data to a new folder
#lfp_recording = recording_final.save(folder='processed_LFP_Mouse2', overwrite = True)
lfp_recording = recording_final

In [ ]:

lfp = sw.plot_traces(recording_final, order_channel_by_depth = True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Define Setup ---
fs_lfp = 30000
pre_time = 0.1
post_time = 0.5
n_pre = int(pre_time * fs_lfp)
n_post = int(post_time * fs_lfp)
total_points = n_pre + n_post
t_vec = np.linspace(-pre_time, post_time, total_points)

# Ensure triggers are scaled correctly (assuming they passed diagnostics)
# If your diagnostics showed they were already scaled, remove the '/ 30'
triggers_lfp = (np.array(triggers_checkerboard[2])).astype(int)
triggers_lfp = triggers_lfp[1:]

n_triggers = len(triggers_lfp)
n_channels = lfp_recording.get_num_channels()

# --- 2. Extract Epochs (Original Order) ---
print("Extracting epochs...")
epochs_raw = np.zeros((n_triggers, n_channels, total_points))

for i, trig in enumerate(triggers_lfp):
    start = trig - n_pre
    end = trig + n_post
    # Basic bounds check
    if start >= 0 and end < lfp_recording.get_num_frames():
        # return_scaled=True for uV units
        chunk = lfp_recording.get_traces(start_frame=start, end_frame=end, return_in_uV=True)
        epochs_raw[i, :, :] = chunk.T
    else:
        # Fill out-of-bounds trials with NaNs so they appear blank in plots
        epochs_raw[i, :, :] = np.nan
        print(f"Warning: Trigger {i} at sample {trig} is out of bounds.")

# --- 3. Reorder by Depth (The New Part) ---
print("Sorting channels by physical depth...")

# Get coordinates (N channels x 2 dimensions [x, y])
# For Neuropixels, Y dimension (index 1) is typically depth along the shank.
locations = lfp_recording.get_channel_locations()
depths_um = locations[:, 1]

# Get indices that sort the depths from deepest (smallest Y) to shallowest (largest Y)
sorted_indices = np.argsort(depths_um)

# Apply this sorting to the channel dimension (axis 1) of the data cube
epochs_sorted = epochs_raw[:, sorted_indices, :]

# Calculate actual depth range for plotting labels
min_depth = np.min(depths_um)
max_depth = np.max(depths_um)

print(f"Epochs reordered. Depth range: {min_depth:.1f} to {max_depth:.1f} um.")
print(f"Final data shape: {epochs_sorted.shape}")

In [ ]:
from ipywidgets import interact, IntSlider
import matplotlib.cm as cm

def plot_single_trigger_depth(trigger_idx):
    """Plots a heatmap of depth-ordered channels for a specific trigger."""
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Use the reordered data
    data = epochs_sorted[trigger_idx]
    
    # Plot Heatmap
    # origin='lower' puts the first index (deepest channel) at the bottom.
    # extent sets the axes units: [t_start, t_end, y_bottom_val, y_top_val]
    # We use channel rank (0 to n_channels) for the Y-axis to ensure a dense plot,
    # but we will label it with actual microns below.
    im = ax.imshow(data, aspect='auto', origin='lower', 
                   extent=[t_vec[0], t_vec[-1], 0, n_channels],
                   cmap='RdBu_r', vmin=-200, vmax=200) # Adjust vmin/vmax as needed for contrast
    
    # Add trigger line
    ax.axvline(0, color='k', linestyle='--', alpha=0.7, linewidth=1)
    
    # Setup axes labels
    ax.set_xlabel('Time relative to trigger (s)')
    
    # Custom Y-axis ticks to show microns instead of rank index
    # Put 5 ticks evenly spaced
    yticks_indices = np.linspace(0, n_channels, 5)
    yticks_labels = np.linspace(min_depth, max_depth, 5).astype(int)
    ax.set_yticks(yticks_indices)
    ax.set_yticklabels(yticks_labels)
    ax.set_ylabel('Approx. Depth (\u03BCm)\n(Bottom=Deep, Top=Shallow)')
    
    ax.set_title(f'LFP Response by Depth - Trigger #{trigger_idx}')
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax, pad=0.02)
    cbar.set_label('Voltage (\u03BCV)')
    
    plt.tight_layout()
    plt.show()

# Create the Slider
# Check if triggers exist before launching widget
if n_triggers > 0:
    interact(plot_single_trigger_depth, 
             trigger_idx=IntSlider(min=0, max=n_triggers-1, step=1, value=0))
else:
    print("No valid triggers found. Cannot generate plot.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# 1. Compute the Average ERP from the sorted data
# Shape: (n_channels, n_timepoints)
# Index 0 = Deepest Channel, Index -1 = Shallowest Channel
erp_sorted = np.nanmean(epochs_sorted, axis=0)

# 2. Setup Plot
fig, ax = plt.subplots(figsize=(10, 14))

# Parameters for visibility
offset_step = 2  # Vertical space between channels (in uV)
scaling = 1.0     # Multiplier to make traces bigger/smaller without changing offset
n_chans_to_plot = erp_sorted.shape[0]

# Generate depth colors (Purple=Deep, Yellow=Shallow)
colors = cm.plasma(np.linspace(0, 1, n_chans_to_plot))

# 3. Plot Loop
# We iterate through channels (0 is deepest)
for i in range(n_chans_to_plot):
    # Calculate Y-position: Channel Index * Offset
    y_pos = i * offset_step
    
    # Scale the voltage trace
    trace = erp_sorted[i, :] * scaling
    
    # Plot (add y_pos to shift it up)
    ax.plot(t_vec, trace + y_pos, color=colors[i], linewidth=0.6)

# 4. Reference Lines
ax.axvline(0, color='k', linestyle='--', alpha=0.5, linewidth=1) # Trigger time
ax.axvline(0.1, color='r', linestyle='--', alpha=0.5, linewidth=1)

# 5. Custom Y-Axis (Map Offsets to Real Depth)
# We place 10 ticks evenly spaced along the plot height
tick_indices = np.linspace(0, n_chans_to_plot - 1, 10).astype(int)
tick_positions = tick_indices * offset_step

# We grab the actual depth (um) corresponding to those indices
# (Using the sorted depths array we created in the previous step)
# sorted_indices was created in the extraction step: depths_um[sorted_indices]
sorted_depths = depths_um[sorted_indices]
tick_labels = sorted_depths[tick_indices].astype(int)

ax.set_yticks(tick_positions)
ax.set_yticklabels(tick_labels)

# 6. Labels and Titles
ax.set_ylabel('Depth (\u03BCm) [Physical Location]')
ax.set_xlabel('Time relative to trigger (s)')
ax.set_title(f'Laminar LFP Profile (Waterfall)\nAvg of {n_triggers} Trials | Bottom=Tip, Top=Base')
ax.set_xlim(t_vec[0], t_vec[-1])

# Remove top/right spines for cleaner look
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Compute the Average LFP (Mean across triggers)
# Shape: (n_channels, n_timepoints)
# Index 0 = Deepest, Index -1 = Shallowest
mean_lfp_map = np.nanmean(epochs_sorted, axis=0)

# 2. Setup the Plot
fig, ax = plt.subplots(figsize=(12, 8))

# 3. Determine Color Scale
# distinct contrast usually requires setting limits. 
# We use the 98th percentile to ignore extreme artifact outliers.
limit = np.nanpercentile(np.abs(mean_lfp_map), 98) 

# 4. Plot Heatmap
# origin='lower' puts the deepest channel (index 0) at the bottom
im = ax.imshow(mean_lfp_map, 
               aspect='auto', 
               origin='lower',
               extent=[t_vec[0], t_vec[-1], 0, n_channels],
               cmap='RdBu_r',  # Red=Positive, Blue=Negative (Standard Ephys)
               vmin=-limit, 
               vmax=limit)

# 5. Configure Y-Axis to show Physical Depth
# We select 10 evenly spaced ticks
tick_indices = np.linspace(0, n_channels - 1, 10).astype(int)

# Map these indices to the actual sorted depth values
# (We use the sorted_depths array derived in the previous step)
sorted_depths = depths_um[sorted_indices]
tick_labels = sorted_depths[tick_indices].astype(int)

ax.set_yticks(tick_indices)
ax.set_yticklabels(tick_labels)
ax.set_ylabel('Depth (\u03BCm) [Physical Location]')
ax.set_xlabel('Time relative to trigger (s)')

# 6. Aesthetics
ax.set_title(f'Average LFP Response (Heatmap)\nn={n_triggers} Trials | Bottom=Tip, Top=Base')
ax.axvline(0, color='k', linestyle='--', linewidth=1.5, label='Trigger')

# Colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Voltage (\u03BCV)')

plt.tight_layout()
plt.show()

In [ ]:
lfp_c_p(
    lfp_recording,
    concat_kilosort["Mouse 2"],
    triggers_checkerboard[2],
    True,
    30,
    1
)


In [ ]:
c = sw.plot_traces(recordings_checkerboard[2], order_channel_by_depth = True)
plt.show()
h = sw.plot_traces(recording_habituation[2], order_channel_by_depth = True)
plt.show()

In [ ]:
%matplotlib widget

cr = sw.plot_rasters(sortings_checkerboard[2])
plt.show()
ch = sw.plot_rasters(sorting_habituation[2])
plt.show()


In [ ]:
sorting_analyzer_checkerboard = si.create_sorting_analyzer(sorting=sortings_checkerboard[2], recording=recordings_checkerboard[2])

# we need to compute some required extensions
sorting_analyzer_checkerboard.compute(['random_spikes', 'templates', 'spike_amplitudes', 'spike_locations', 'noise_levels', 'quality_metrics'])
# note that spike_locations are optional, but recommended to compute accurate spike depths

# optionally, we can pass an LFP recording to compute RMS/PSD in the LFP band
recording_lfp_check = spre.bandpass_filter(recordings_checkerboard[2], freq_min=1, freq_max=300)
# we can also decimate the LFP to speed up the process
recording_lfp_check = spre.decimate(recording_lfp_check, 10)

sorting_analyzer_habituation = si.create_sorting_analyzer(sorting=sorting_habituation[2], recording=recording_habituation[2])

# we need to compute some required extensions
sorting_analyzer_habituation.compute(['random_spikes', 'templates', 'spike_amplitudes', 'spike_locations', 'noise_levels', 'quality_metrics'])
# note that spike_locations are optional, but recommended to compute accurate spike depths

# optionally, we can pass an LFP recording to compute RMS/PSD in the LFP band
recording_lfp_hab = spre.bandpass_filter(recording_habituation[2], freq_min=1, freq_max=300)
# we can also decimate the LFP to speed up the process
recording_lfp_hab = spre.decimate(recording_lfp_hab, 10)




In [ ]:
sexp.export_to_ibl_gui(
    sorting_analyzer=sorting_analyzer_checkerboard,
    output_folder=r'Z:\Saij\ephys\Mouse 2\checkerboard',
    lfp_recording=recording_lfp_check,
    n_jobs=-1
)

print("done")

sexp.export_to_ibl_gui(
    sorting_analyzer=sorting_analyzer_habituation,
    output_folder=r'Z:\Saij\ephys\Mouse 2\habituation',
    lfp_recording=recording_lfp_hab,
    n_jobs=-1
)

print("done")




In [ ]:
sw.plot_drift_raster_map(
    sorting_analyzer=sorting_analyzer_checkerboard,
    scatter_decimate=10, 
    backend='matplotlib'
)


In [ ]:
sw.plot_drift_raster_map(
    sorting_analyzer=sorting_analyzer_habituation,
    scatter_decimate=10, 
    backend='matplotlib'
)




In [ ]:
%matplotlib widget

import spikeinterface.widgets as sw
sw.plot_unit_probe_map(sorting_analyzer = sorting_analyzer_checkerboard,
                            animated = False,
                            with_channel_ids = False)


In [ ]:
plt.show()

In [ ]:
recording = recordings_checkerboard[2]
sorting = sortings_checkerboard[2]
sorting_analyzer = sorting_analyzer_checkerboard

In [ ]:
import spikeinterface as si
import numpy as np

# Assuming 'recording' is your loaded object
print("Channel IDs:", recording.get_channel_ids()[:10])

# Check locations
locations = recording.get_channel_locations()
print("First 5 locations (x, y):\n", locations[:5])

# Check bounds
print("X range:", np.min(locations[:, 0]), np.max(locations[:, 0]))
print("Y range:", np.min(locations[:, 1]), np.max(locations[:, 1]))

In [ ]:
import numpy as np
import probeinterface as pi

# 1. Load the KiloSort geometry files
# flattened to ensure 1D arrays
chan_map = np.load(r'Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Concatenated Data\kilosort4\channel_map.npy').flatten() 
chan_pos = np.load(r'Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Concatenated Data\kilosort4\channel_positions.npy')

# 2. FIX: Recenter the X-coordinates
# This moves Shank 3 (x=758) back to x=0 so it plots normally
print(f"Original X range: {np.min(chan_pos[:,0])} - {np.max(chan_pos[:,0])}")
chan_pos[:, 0] -= np.min(chan_pos[:, 0])
print(f"New X range: {np.min(chan_pos[:,0])} - {np.max(chan_pos[:,0])}")

# 3. Create the Probe
probe = pi.Probe(ndim=2, si_units='um')
probe.set_contacts(positions=chan_pos, shapes='circle', shape_params={'radius': 6})

# 4. CRITICAL: Link Probe to Recording IDs
# If your recording.get_channel_ids() are [0, 1, 2...], 
# you typically want the probe indices to match that simple indexing.
# However, we must trust the 'chan_map' file which tells us which geometric 
# spot corresponds to which data row.
if np.array_equal(recording.get_channel_ids(), np.arange(recording.get_num_channels())):
    # If recording has simple IDs (0, 1, 2...), use the map as indices
    probe.set_device_channel_indices(chan_map)
else:
    # If recording has complex IDs (e.g. "AP0", "AP1"), we need to map differently.
    # For now, assuming simple integer IDs from your snippet:
    probe.set_device_channel_indices(chan_map)

# 5. Attach and Verify
recording_geo = recording.set_probe(probe)

# Quick visual check - this should now look like one nice shank, not 5 columns
pi.plotting.plot_probe(recording_geo.get_probe(), with_contact_id=True)

In [ ]:
plt.show()

In [ ]:
# Create a new analyzer with the corrected geometry
new_analyzer = si.create_sorting_analyzer(sorting, recording_geo, format="memory")
new_analyzer.compute(['random_spikes', 'templates', 'spike_amplitudes', 'spike_locations', 'noise_levels', 'quality_metrics'])





In [ ]:
sw.plot_drift_raster_map(
    sorting_analyzer= new_analyzer,
    scatter_decimate=10, 
    backend='matplotlib'
)


In [ ]:
sw.plot_unit_probe_map(sorting_analyzer = new_analyzer,
                            animated = False,
                            with_channel_ids = False)




In [ ]:
plt.show()


In [ ]:
# Plot the template of the first unit across all channels
# If you see "noise" instead of a spike shape, your channel_map is likely shuffled.
sw.plot_traces(recording_geo, order_channel_by_depth = True)

In [ ]:
sw.plot_traces(recording_geo, order_channel_by_depth = True)
plt.show()

In [ ]:
import numpy as np
import probeinterface as pi
import spikeinterface as si

# 1. Load the geometry files

chan_map = np.load(r'Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Concatenated Data\kilosort4\channel_map.npy').flatten() 
chan_pos = np.load(r'Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Concatenated Data\kilosort4\channel_positions.npy')



# 2. FIX 1: Recenter X-coordinates (Remove the ~750um offset)
# This ensures the shank plots in the center, not far to the right
chan_pos[:, 0] -= np.min(chan_pos[:, 0])

# 3. Create the Probe
probe = pi.Probe(ndim=2, si_units='um')
# Neuropixels contacts are 12um x 12um squares or circles. 
# Using slightly smaller circles (radius 5-6) helps visualization.
probe.set_contacts(positions=chan_pos, shapes='circle', shape_params={'radius': 6})

# 4. FIX 2: FORCE SINGLE SHANK (The critical fix for your 5-column issue)
# We overwrite any existing shank info by setting all IDs to 0
probe.set_shank_ids(np.zeros(len(chan_map), dtype=int))

# 5. FIX 3: Link Device Indices
# This ensures the binary data rows match the correct geometric positions
probe.set_device_channel_indices(chan_map)

# 6. Attach to recording
recording_geo = recording.set_probe(probe)

# 7. Verify Visual
# This should now show ONE column labeled "0"
pi.plotting.plot_probe(recording_geo.get_probe(), with_contact_id=True)
plt.show()

In [ ]:
import numpy as np
import probeinterface as pi
import matplotlib.pyplot as plt

# 1. RELOAD FRESH (Prevents the double-subtraction error)
chan_map = np.load(r'Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Concatenated Data\kilosort4\channel_map.npy').flatten() 
chan_pos = np.load(r'Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Concatenated Data\kilosort4\channel_positions.npy')




print(f"Loaded {len(chan_map)} channels.")
print(f"Original X Range: {chan_pos[:,0].min()} to {chan_pos[:,0].max()}")
print(f"Original Y Range: {chan_pos[:,1].min()} to {chan_pos[:,1].max()}")

# 2. Safety Check: Shape Mismatch
# KiloSort results sometimes have 384 positions but fewer map indices. 
# We ensure we only use the positions that correspond to our map.
if chan_pos.shape[0] != chan_map.shape[0]:
    print("⚠️ Shape Mismatch detected! Attempting to slice positions...")
    # NOTE: This assumes chan_pos was the full probe and chan_map indexes into it.
    # If chan_pos matches chan_map 1-to-1, this block is skipped.
    if chan_pos.shape[0] > chan_map.shape[0]:
         # Common KS output issue: slice to match
         chan_pos = chan_pos[chan_map] 

# 3. FIX: Recenter X coordinates to 0
chan_pos[:, 0] -= np.min(chan_pos[:, 0])

# 4. Construct Probe
probe = pi.Probe(ndim=2, si_units='um')
probe.set_contacts(positions=chan_pos, shapes='circle', shape_params={'radius': 6})

# 5. FORCE SINGLE SHANK (Fixes the 5-column split)
probe.set_shank_ids(np.zeros(len(chan_map), dtype=int))

# 6. Map Device Indices
probe.set_device_channel_indices(chan_map)

# 7. Plot with Explicit Limits
# We force the plot to look at where the data actually IS
plt.figure(figsize=(5, 15))
pi.plotting.plot_probe(probe, with_contact_id=True, with_device_index=False)

# Force the camera to look at the probe
plt.ylim(np.min(chan_pos[:,1]) - 50, np.max(chan_pos[:,1]) + 50)
plt.xlim(-50, 100) # Since we zeroed X, it should be between 0 and ~60
plt.title("Corrected Single-Shank Probe")
plt.show()

In [ ]:
# Only run this if the plot above looked good
recording_geo = recording.set_probe(probe)

# Double check the recording object accepted it
print(recording_geo.get_probe()) 
# Should say: "Probe - 1 shanks - XXX contacts"

In [ ]:
new_analyzer = si.create_sorting_analyzer(sorting, recording_geo, format="memory")
new_analyzer.compute(['random_spikes', 'templates', 'spike_amplitudes', 'spike_locations', 'noise_levels', 'quality_metrics'])







In [ ]:
sw.plot_drift_raster_map(
    sorting_analyzer= new_analyzer,
    scatter_decimate=10, 
    backend='matplotlib'
)
plt.show()
sw.plot_unit_probe_map(sorting_analyzer = new_analyzer,
                            animated = False,
                            with_channel_ids = False)

plt.show()




In [ ]:
new_analyzer.compute("unit_locations")
sw.plot_unit_depths(new_analyzer)
plt.show()

In [ ]:
sw.plot_drift_raster_map(
    sorting_analyzer= new_analyzer,
    scatter_decimate=10, 
    backend='matplotlib'
)
plt.show()


In [ ]:
chan_map = np.load(r'Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Concatenated Data\kilosort4\channel_map.npy').flatten() 
chan_pos = np.load(r'Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Concatenated Data\kilosort4\channel_positions.npy')

recording= se.read_binary(
    file_paths = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Concatenated Data\mouse2_renewal_concatenated_neural_data.dat", 
    dtype = dtype, 
    sampling_frequency= fs, 
    num_channels= num_channels, 
    gain_to_uV= 0.1949999928474426, 
    offset_to_uV= 0)

sorting = se.read_kilosort(r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Concatenated Data\kilosort4")


df_pos = pd.DataFrame(chan_pos)
df_pos = df_pos.rename(columns = {0: "x", 1: "y"}, errors = "raise")
df_pos

df_ids = pd.DataFrame(chan_map)
df_ids = df_ids.rename(columns = {0: "channel_id"})
df_ids

channels = pd.concat([df_pos, df_ids], axis = 1)
channels




In [ ]:
df = channels
import probeinterface as pi
import numpy as np
import pandas as pd

# 1. Extract data from your DataFrame
# Ensure x and y are in a (num_contacts, 2) numpy array
positions = df[['x', 'y']].to_numpy()

# Ensure channel_ids are a 1D array/list
device_channel_indices = df['channel_id'].to_numpy()

# 2. Create the Probe object
# ndim=2 because Neuropixels are planar probes
probe = pi.Probe(ndim=2, si_units='um')

# 3. Set the geometric contacts
# Neuropixels 2.0 contacts are 12x12 um squares.
probe.set_contacts(
    positions=positions, 
    shapes='square', 
    shape_params={'width': 12}
)

# 4. Map the geometry to your specific channel IDs
probe.set_device_channel_indices(device_channel_indices)

# 5. (Optional) Set Probe Metadata for provenance
probe.annotate(
    name='Neuropixels 2.0', 
    manufacturer='Imec', 
    description='Custom layout from DataFrame'
)

# 6. Create a ProbeGroup (required for saving/attaching to recordings)
probegroup = pi.ProbeGroup()
probegroup.add_probe(probe)

# 7. Visualization check
pi.plotting.plot_probe(probe, with_contact_id=True, with_device_index = True)

In [ ]:
print(device_channel_indices)

In [ ]:
p = probe.to_dataframe()
print(p)